In [33]:
#Cell 1
import os
import json
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.pipeline import Pipeline

In [34]:
#Cell 2
os.makedirs("RF_hyperparametertuning", exist_ok=True)

DATA_PATH = "dataset/EVSE-B-PowerCombined_filtered.csv"
OUTPUT_CSV = "RF_hyperparametertuning/rf_two_stage_results.csv"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (49017, 10)


,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface
0,12/25/2023 22:35,978,5.165,1027,5300,idle,syn-flood,DoS,attack,ocpp
1,12/25/2023 22:35,872,5.161,1009,4980,idle,syn-flood,DoS,attack,ocpp
2,12/25/2023 22:35,1017,5.165,1029,5300,idle,syn-flood,DoS,attack,ocpp
3,12/25/2023 22:35,930,5.161,1005,5180,idle,syn-flood,DoS,attack,ocpp
4,12/25/2023 22:35,958,5.165,1034,5180,idle,syn-flood,DoS,attack,ocpp


In [35]:
#Cell 3
drop_cols = ["Attack", "Attack-Group", "Label", "interface", "time"]

def prep_xy(df_in):
    X = df_in.drop(columns=drop_cols, errors="ignore")
    y = df_in["Attack"]
    X = pd.get_dummies(X, drop_first=False)
    return X, y

In [36]:
#Cell 4
df_train, df_temp = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["Attack"] if "Attack" in df.columns else None
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    random_state=42,
    stratify=df_temp["Attack"] if "Attack" in df_temp.columns else None
)

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)

Train shape: (34311, 10)
Validation shape: (7353, 10)
Test shape: (7353, 10)


In [37]:
#Cell 5
X_train, y_train = prep_xy(df_train)
X_val, y_val = prep_xy(df_val)
X_test, y_test = prep_xy(df_test)

X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (34311, 6)
X_val: (7353, 6)
X_test: (7353, 6)


In [38]:
#Cell 6
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

def build_model(params):
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(random_state=42, n_jobs=-1))
    ])
    model.set_params(**params)
    return model

def append_result(row_dict):
    pd.DataFrame([row_dict]).to_csv(
        OUTPUT_CSV,
        mode="a",
        header=not os.path.exists(OUTPUT_CSV),
        index=False
    )

def run_stage(stage_name, param_grid, X_tr, y_tr, X_va, y_va):
    results = []

    for run_idx, params in enumerate(ParameterGrid(param_grid), start=1):
        model = build_model(params)
        model.fit(X_tr, y_tr)

        val_pred = model.predict(X_va)
        val_acc = accuracy_score(y_va, val_pred)

        row = {
            "stage": stage_name,
            "run": run_idx,
            **params,
            "val_accuracy": val_acc
        }

        results.append(row)
        append_result(row)

        print(f"{stage_name} | run {run_idx} saved | val_accuracy={val_acc:.6f}")

    return pd.DataFrame(results).sort_values("val_accuracy", ascending=False).reset_index(drop=True)

In [ ]:
#Cell 7
stage1_grid = {
    "rf__n_estimators": [100, 200, 300, 500, 800],
    "rf__max_depth": [None, 5, 10, 20, 40, 60],
    "rf__min_samples_split": [2, 5, 10, 15, 20],
    "rf__min_samples_leaf": [1, 2, 4, 8, 12],
    "rf__max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "rf__bootstrap": [True, False],
    "rf__criterion": ["gini", "entropy", "log_loss"],
    "rf__class_weight": [None, "balanced", "balanced_subsample"],
    "rf__min_impurity_decrease": [0.0, 1e-4, 1e-3],
    "rf__max_samples": [None, 0.7, 0.9],
    "rf__ccp_alpha": [0.0, 1e-4, 1e-3],
}
print("Starting Stage 1 coarse search...")
stage1_results = run_stage("stage1", stage1_grid, X_train, y_train, X_val, y_val)

print("\nTop Stage 1 results:")
stage1_results.head(10)

Starting Stage 1 coarse search...
stage1 | run 1 saved | val_accuracy=0.860465
stage1 | run 2 saved | val_accuracy=0.861961
stage1 | run 3 saved | val_accuracy=0.861553
stage1 | run 4 saved | val_accuracy=0.862369
stage1 | run 5 saved | val_accuracy=0.861417
stage1 | run 6 saved | val_accuracy=0.865905
stage1 | run 7 saved | val_accuracy=0.865225
stage1 | run 8 saved | val_accuracy=0.864273
stage1 | run 9 saved | val_accuracy=0.865361
stage1 | run 10 saved | val_accuracy=0.866041
stage1 | run 11 saved | val_accuracy=0.862913
stage1 | run 12 saved | val_accuracy=0.864137
stage1 | run 13 saved | val_accuracy=0.864817
stage1 | run 14 saved | val_accuracy=0.864817
stage1 | run 15 saved | val_accuracy=0.864545
stage1 | run 16 saved | val_accuracy=0.866449
stage1 | run 17 saved | val_accuracy=0.865225
stage1 | run 18 saved | val_accuracy=0.865633
stage1 | run 19 saved | val_accuracy=0.866313
stage1 | run 20 saved | val_accuracy=0.865769
stage1 | run 21 saved | val_accuracy=0.867537
stage1 | 

In [ ]:
#Cell 8
if stage1_results.empty:
    raise ValueError("Stage 1 produced no results.")

best_stage1 = stage1_results.iloc[0].to_dict()

print("Best Stage 1 configuration:")
print(json.dumps(best_stage1, indent=2, default=str))

In [ ]:
#Cell 9
def make_stage2_grid(best_row):
    n_estimators = int(best_row["rf__n_estimators"])
    max_depth = best_row["rf__max_depth"]
    min_split = int(best_row["rf__min_samples_split"])
    min_leaf = int(best_row["rf__min_samples_leaf"])

    n_estimators_list = sorted(set([
        max(50, n_estimators - 50),
        n_estimators,
        n_estimators + 50
    ]))

    if max_depth is None:
        max_depth_list = [None, 20, 40]
    else:
        max_depth_list = sorted(set([
            max(5, int(max_depth) - 10),
            int(max_depth),
            int(max_depth) + 10
        ]))

    min_split_list = sorted(set([
        max(2, min_split - 2),
        min_split,
        min_split + 2
    ]))

    min_leaf_list = sorted(set([
        max(1, min_leaf - 1),
        min_leaf,
        min_leaf + 1
    ]))

    return {
        "rf__n_estimators": n_estimators_list,
        "rf__max_depth": max_depth_list,
        "rf__min_samples_split": min_split_list,
        "rf__min_samples_leaf": min_leaf_list,
        "rf__max_features": ["sqrt", "log2"],
        "rf__bootstrap": [True],
    }

stage2_grid = make_stage2_grid(best_stage1)
stage2_grid

In [ ]:
#Cell 10
print("Starting Stage 2 fine search...")
stage2_results = run_stage("stage2", stage2_grid, X_train, y_train, X_val, y_val)

print("\nTop Stage 2 results:")
stage2_results.head(10)

In [ ]:
#Cell 11
all_results = pd.concat([stage1_results, stage2_results], ignore_index=True)
all_results = all_results.sort_values("val_accuracy", ascending=False).reset_index(drop=True)

all_results.to_csv(OUTPUT_CSV, index=False)

print("All results saved to:", OUTPUT_CSV)
all_results.head(20)

In [ ]:
#Cell 12
from sklearn.metrics import log_loss, f1_score
best_overall = all_results.iloc[0].to_dict()

print("Best overall configuration:")
print(json.dumps(best_overall, indent=2, default=str))

In [ ]:
best_params = {k: v for k, v in best_overall.items() if k.startswith("rf__")}

if best_params.get("rf__max_depth") is not None:
    best_params["rf__max_depth"] = int(best_params["rf__max_depth"])
best_params["rf__n_estimators"] = int(best_params["rf__n_estimators"])
best_params["rf__min_samples_split"] = int(best_params["rf__min_samples_split"])
best_params["rf__min_samples_leaf"] = int(best_params["rf__min_samples_leaf"])
best_params["rf__bootstrap"] = bool(best_params["rf__bootstrap"])
if "rf__max_features" in best_params and isinstance(best_params["rf__max_features"], float):
    best_params["rf__max_features"] = float(best_params["rf__max_features"])

final_model = build_model(best_params)
final_model.fit(
    pd.concat([X_train, X_val], axis=0),
    pd.concat([y_train, y_val], axis=0)
)

val_proba = final_model.predict_proba(X_val)
test_proba = final_model.predict_proba(X_test)

val_pred = final_model.predict(X_val)
test_pred = final_model.predict(X_test)

best_val_loss = log_loss(y_val, val_proba)
test_loss = log_loss(y_test, test_proba)
macro_f1 = f1_score(y_test, test_pred, average="macro")

print("best_val_loss:", best_val_loss)
print("test_loss:", test_loss)
print("macro_f1:", macro_f1)
print("Test accuracy:", accuracy_score(y_test, test_pred))
print(classification_report(y_test, test_pred))

In [ ]:
#Cell 14
results_check = pd.read_csv(OUTPUT_CSV)
results_check.tail(20)